# Tâche 7: Contrôle qualité du corpus

**Projet 1 · African Language Data & LLM Readiness**

**Consigne** : détecter les doublons exacts et quasi-doublons, documents vides/courts, encodage, boilerplate, inadéquation de langue → champ `quality_flags`.

Le code est dans `src/controle_qualite.py`. Il lit le corpus normalisé (tâche 6, dossier indiqué dans `config/config.yaml`, section `qualite`), sans jamais le modifier ni le recopier.

**Aucun texte n'est retiré** : un problème détecté devient un drapeau dans la provenance (`quality_flags`), parmi quatre valeurs : `near_duplicate`, `template_generated`, `boilerplate`, `language_mismatch`.

| Sous-étape | Contenu | État |
|---|---|---|
| 7.1 | Doublons exacts entre langues (comptés, pas écartés) | fait : 85 textes dans 38 groupes |
| 7.2 | Quasi-doublons → `near_duplicate` (même article seulement) | validé le 18/11 : 7 106 textes (0,18 %) |
| 7.3 | Ébauches en série → `template_generated` | validé le 19/11 : 488 083 textes (12,4 %) |
| 7.4 | Texte répété → `boilerplate` | validé le 19/11 : 291 967 textes (7,4 %) |
| 7.5 | Mauvaise langue → `language_mismatch` (GlotLID v3) ; chaque langue jugée sur ses seules étiquettes | analyse en cours |
| 7.6 | Textes courts, encodage, messages d'erreur (comptés, sans drapeau) | validé le 19/11 : 2 vraies erreurs de collecte (ha, ig), écartées à l'export (tâche 8) |
| 7.7 | Run complet et contrôles | à faire |

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [7]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ANALYSES = ROOT / "data" / "processed" / "qualite" / "analyses"


def lancer(*args):
    # Lance src/controle_qualite.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "controle_qualite.py"), *args], cwd=ROOT, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("\nArrêté.")
    return p.wait()


def derniere(nom):
    # Dernière analyse d'un type donné (doublons_langues, sondage_langues…)
    return sorted(ANALYSES.glob(f"*_{nom}"))[-1]

## 7.1: Doublons exacts entre langues

Dans chaque langue, les doublons exacts sont déjà retirés (règle T6). On cherche ici les textes identiques présents dans **deux éditions différentes**, à partir des empreintes SHA-256 de la provenance. Ils sont comptés et listés, pas écartés. Quelques minutes.

Résultat : `bilan.json` (comptes par paire de langues) et `doublons_langues.jsonl` (chaque groupe avec un extrait et les adresses des sources).

In [9]:
lancer("--doublons-langues")
d = derniere("doublons_langues")
bilan = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print({k: bilan[k] for k in ("textes_lus", "groupes", "textes_concernes", "groupes_dans_une_meme_langue", "mots_par_texte")})
pd.Series(bilan["par_paire"], name="groupes").head(20)

2026-11-17T08:52:47 INFO 7.1 Doublons exacts entre langues : 69 langues, source data\processed\normalise\20261113T141439Z
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still failed after 10 retries, giving up.
OpenBLAS error: Memory allocation still fa

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\akota\\desktop\\projet_stage_nlp\\data\\processed\\qualite\\analyses\\20261117T085247Z_doublons_langues\\bilan.json'

In [4]:
# Exemples : les 15 premiers groupes
pd.read_json(d / "doublons_langues.jsonl", lines=True).head(15)

,langues,mots,caracteres,extrait,sources
0,"[rn, tn]",1,3,[1],{'rn': 'https://rn.wikipedia.org/w/index.php?o...
1,"[ar, arz]",6,27,"قد يقصد من ""مقاطعة فاييت"" :",{'ar': 'https://ar.wikipedia.org/w/index.php?o...
2,"[mg, mos]",11,68,Wonder Woman. DC Comics. 1941. William Moulton...,{'mg': 'https://mg.wikipedia.org/w/index.php?o...
3,"[ar, arz]",6,27,"قد يقصد من ""مقاطعة كارول"" :",{'ar': 'https://ar.wikipedia.org/w/index.php?o...
4,"[ha, ig]",11,91,upstream connect error or disconnect/reset bef...,{'ha': 'https://ha.wikipedia.org/w/index.php?o...
5,"[ar, arz]",6,27,"قد يقصد من ""مقاطعة كلارك"" :",{'ar': 'https://ar.wikipedia.org/w/index.php?o...
6,"[nso, tum]",2,17,Edit instructions,{'nso': 'https://nso.wikipedia.org/w/index.php...
7,"[ee, sg]",1,6,Kpakpa,{'ee': 'https://ee.wikipedia.org/w/index.php?o...
8,"[ha, rw]",106,551,Stanley Fabian Allotey (an haife shi a ranar 1...,{'ha': 'https://ha.wikipedia.org/w/index.php?o...
9,"[xh, zu]",64,596,UJosie Mpama (1903 - 1979) wazalelwa ePotchefs...,{'xh': 'https://xh.wikipedia.org/w/index.php?o...


## Préparation de 7.5: GlotLID v3 et sondage des langues

Installe le modèle d'identification de langue GlotLID v3 (`cis-lmu/glotlid`, fichier `model_v3.bin`, 1,69 Go, licence Apache-2.0). Il est téléchargé **une seule fois** dans le cache Hugging Face de l'ordinateur, hors du projet.

Puis tire 100 paragraphes d'au moins 10 mots par langue et regarde quelle langue le modèle y reconnaît. Ce sondage sert à vérifier l'installation et à régler la liste des langues proches avant 7.5. Le premier lancement dépend surtout du téléchargement.

In [5]:
lancer("--sonder-langues")
d = derniere("sondage_langues")
s = json.loads((d / "sondage_langues.json").read_text(encoding="utf-8"))
tab = pd.DataFrame([{"langue": r["langue"], "attendue": ", ".join(r["etiquettes_attendues"]) or "absente du modèle",
                     "paragraphes": r["paragraphes"], "dans la langue attendue": r["dans_la_langue_attendue"],
                     "trouvé": ", ".join(f"{k} {v}" for k, v in list(r["etiquettes_trouvees"].items())[:3])}
                    for r in s["langues"]]).set_index("langue")
pd.set_option("display.max_rows", 100)
tab

2026-11-13T17:28:36 INFO Tirage de 100 paragraphes d'au moins 10 mots par langue…
2026-11-13T17:29:38 INFO Téléchargement (1,69 Go, une seule fois) ou chargement de GlotLID (cis-lmu/glotlid, model_v3.bin)…
C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:773: UserWarning: Not enough free disk space to download the file. The expected file size is: 1687.09 MB. The target location C:\Users\akota\.cache\huggingface\hub\models--cis-lmu--glotlid\blobs only has 151.52 MB free disk space.
  warnings.warn(
2026-11-13T17:35:45 INFO Modèle chargé : C:\Users\akota\.cache\huggingface\hub\models--cis-lmu--glotlid\snapshots\e9592e0a9bc3b6f2a946de94faaa188d1dd6f522\model_v3.bin
2026-11-13T17:35:45 INFO aa     0 paragraphes,   0 dans la langue attendue (langue absente du modèle) ; trouvé : 
2026-11-13T17:35:45 INFO af   100 paragraphes,  91 dans la langue attendue ['afr_Latn'] ; trouvé : afr_Latn 91, eng_Latn 3, deu_Latn 2
2026-11-13T17:35:45 INF

,attendue,paragraphes,dans la langue attendue,trouvé
langue,,,,
aa,absente du modèle,0,0,
af,afr_Latn,100,91,"afr_Latn 91, eng_Latn 3, deu_Latn 2"
ak,"twi_Latn, fat_Latn",1,0,eng_Latn 1
am,amh_Ethi,100,87,"amh_Ethi 87, yor_Latn 5, eng_Latn 3"
ann,ann_Latn,100,99,"ann_Latn 99, eng_Latn 1"
ar,arb_Arab,100,85,"arb_Arab 85, ary_Arab 8, arz_Arab 5"
ary,ary_Arab,100,95,"ary_Arab 95, acm_Arab 2, fra_Latn 1"
arz,arz_Arab,100,93,"arz_Arab 93, arb_Arab 7"
bm,bam_Latn,100,92,"bam_Latn 92, fra_Latn 3, hau_Latn 2"


## 7.2: Quasi-doublons -> `near_duplicate`

Dans chaque langue, on compare les textes par groupes de 5 mots qui se suivent (en minuscules). La méthode MinHash-LSH (128 fonctions de hachage, 16 bandes) trouve les paires candidates sans comparer tous les textes deux à deux ; chaque paire est ensuite confirmée par le calcul exact de la similarité de Jaccard (part des groupes de 5 mots en commun).

**Règle (option B, validée le 17/11)** : une paire est un quasi-doublon si sa similarité est d'au moins **0,8** et si les deux textes parlent **du même sujet** : ils commencent par les mêmes 3 mots, ou **un seul** des deux a une ligne de titre courte (6 mots au plus) ajoutée devant et commence, après ce titre, comme l'autre. Dans chaque groupe de quasi-doublons, le texte le plus long reste sans drapeau, les autres seront marqués `near_duplicate`.

Les paires à 0,8 ou plus dont le sujet diffère sont des **ébauches faites sur un même modèle** où seul le nom change (« même modèle ») : elles sont seulement comptées ici et seront traitées en 7.3 (`template_generated`).

Analyses précédentes : 16/11 (seuil seul) 173 349 textes (4,4 %), surtout des ébauches « même modèle » ; 17/11 (mêmes 3 mots ou texte contenu à 98 % dans l'autre) 7 845 textes (0,20 %), mais la règle « contenu à 98 % » laissait passer des ébauches dont un nom contient l'autre (« Smith County » / « Deaf Smith County ») : elle est remplacée par la ligne de titre (18/11, v0.4). La v0.4 acceptait aussi deux textes ayant chacun leur ligne courte suivie du même corps (villes françaises en swahili : 788 marqués) : corrigé en v0.4.1, analyse retenue du 18/11 07:36 : **7 106 textes (0,18 %)**, 7.2 validée.

Cette analyse ne modifie rien dans la provenance. Environ 20 minutes.

In [3]:
lancer("--quasi-doublons")
d = derniere("quasi_doublons")
qd = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{qd['textes_marques']} textes marqués near_duplicate sur {qd['textes']} "
      f"({100 * qd['textes_marques'] / qd['textes']:.2f} %) ; {qd.get('paires_meme_modele', 0)} paires « même modèle » ; {qd['duree_s']} s")
tab = pd.DataFrame([{"langue": r["langue"], "textes": r["textes"], "marqués": r["textes_marques"],
                     "% marqués": round(100 * r["textes_marques"] / r["textes"], 2) if r["textes"] else 0,
                     "groupes": r["groupes"], "plus grand groupe": r.get("taille_max_groupe", 0),
                     "paires même article": r.get("paires_meme_article", 0),
                     "paires même modèle": r.get("paires_meme_modele", 0)} for r in qd["langues"]]).set_index("langue")
tab.sort_values("% marqués", ascending=False)

2026-11-18T07:36:10 INFO 7.2 Quasi-doublons : 69 langue(s), seuil 0.8, bardeaux de 5 mots, 15 processus
2026-11-18T07:36:30 INFO zgh     12192 textes, paires >= 0.8 :     31 même article,      76 même modèle ;     15 groupes,     31 marqués (19 s)
2026-11-18T07:36:48 INFO ary     20321 textes, paires >= 0.8 :    316 même article,      30 même modèle ;    183 groupes,    254 marqués (38 s)
2026-11-18T07:36:51 INFO so      15572 textes, paires >= 0.8 :     25 même article,     190 même modèle ;     13 groupes,     19 marqués (41 s)
2026-11-18T07:36:54 INFO am      16286 textes, paires >= 0.8 :     14 même article,     805 même modèle ;     14 groupes,     14 marqués (24 s)
2026-11-18T07:36:55 INFO tn       7309 textes, paires >= 0.8 :     17 même article,       0 même modèle ;     17 groupes,     17 marqués (45 s)
2026-11-18T07:37:12 INFO yo      41803 textes, paires >= 0.8 :     21 même article,      33 même modèle ;     21 groupes,     21 marqués (62 s)
2026-11-18T07:37:18 INFO rw     

,textes,marqués,% marqués,groupes,plus grand groupe,paires même article,paires même modèle
langue,,,,,,,
kg,2256,46,2.04,25,20,81,6
nso,9510,185,1.95,73,5,323,4018
ary,20321,254,1.25,183,7,316,30
tig,385,4,1.04,4,2,4,1
ve,1295,13,1.00,13,2,13,596
...,...,...,...,...,...,...,...
kaj,996,0,0.00,0,0,0,0
sg,505,0,0.00,0,0,0,0
pcm,1902,0,0.00,0,0,0,0


In [4]:
# Exemples à relire : paires « même article » par tranche de similarité, puis paires « même modèle » (>= 0,9)
def exemples(*langues):
    for r in qd["langues"]:
        if r["langue"] in langues:
            for e in r["exemples"]:
                print(f"=== {r['langue']}  MÊME ARTICLE  {e['tranche']}  Jaccard {e['jaccard']}")
                print("A :", e["a"][:300].replace("\n", " / "))
                print("B :", e["b"][:300].replace("\n", " / "))
                print("  ", e["source_a"], "|", e["source_b"])
            for e in r.get("exemples_meme_modele", []):
                print(f"--- {r['langue']}  même modèle  Jaccard {e['jaccard']}")
                print("A :", e["a"][:200].replace("\n", " / "))
                print("B :", e["b"][:200].replace("\n", " / "))

exemples("wo", "tw", "ee", "fon", "yo", "ha")

=== ee  MÊME ARTICLE  0.95-1.00  Jaccard 1.0
A : samuel Adebayo Abe nye Anglikan Bisiɔp le Ekiti le ƒe 2007 me, le Ondo Nutome le Nigeria Sɔlemeha la me. /  / Abe xɔ dzudzɔ le dɔme le ƒe 2012 me abe Ekiti ƒe Bisiɔp kple Ondo Nutome ƒe / Bisiɔpgã ene; Chris Omotunde va xɔ ɖe eteƒe wònye Ekiti ƒe Bisiɔp, eye George / Olatunji Lasebikan va xɔ ɖe eteƒe wònye 
B : samuel Adebayo Abe nye Anglikan Bisiɔp le Ekiti le ƒe 2007 me, le Ondo Nutome le Nigeria Sɔlemeha la me. /  / Abe xɔ dzudzɔ le dɔme le ƒe 2012 me abe Ekiti ƒe Bisiɔp kple Ondo Nutome ƒe Bisiɔpgã ene; Chris Omotunde va xɔ ɖe eteƒe wònye Ekiti ƒe Bisiɔp, eye George Olatunji Lasebikan va xɔ ɖe eteƒe wònye 
   https://ee.wikipedia.org/w/index.php?oldid=54134 | https://ee.wikipedia.org/w/index.php?oldid=54218
=== ee  MÊME ARTICLE  0.95-1.00  Jaccard 1.0
A : Lawrence Kpabitey Kodjiku (30 Siamlɔm 1931 – 2022) nye Ghana asrafo kple dunyahela. Enye National Service Secretariat ƒe Dɔdzikpɔla gbãtɔ tso ƒe 1973 va ɖo ƒe 1975,Ashanti Nuto me ƒ

In [5]:
exemples("ar", "arz", "sw", "mg", "af")

=== af  MÊME ARTICLE  0.80-0.90  Jaccard 0.871
A : Lys van outeurs volgens naam: /  / A – B – / C – D – / E – F – / G – H – / I – J – / K – L – / M – N – / O – P – / Q – R – / S – T – / U – V – / W – X – / Y – Z /  / Anna Quindlen
B : Lys van outeurs volgens naam: /  / A – B – / C – D – / E – F – / G – H – / I – J – / K – L – / M – N – / O – P – / Q – R – / S – T – / U – V – / W – X – / Y – Z /  / Qiu Xiaolong
   https://af.wikipedia.org/w/index.php?oldid=1448668 | https://af.wikipedia.org/w/index.php?oldid=2384111
=== af  MÊME ARTICLE  0.80-0.90  Jaccard 0.815
A : Hierdie Doringrivier vloei in die Wes-Kaap. Vir ander riviere met dieselfde naam, sien die dubbelsinnigheidsbladsy by Doringrivier. /  / Die Doringrivier is 'n rivier wat in die Wes-Kaap van Suid-Afrika vloei. Dit is 'n sytak van die Bergrivier. /  / Eksterne skakel / Google-kaart van die Doringrivier se mond
B : Hierdie Doringrivier vloei in die Wes-Kaap. Vir ander riviere met dieselfde naam, sien die dubbelsinnigheidsblads

## 7.3 + 7.4 + 7.6: Ébauches en série, texte répété et contrôles sans drapeau (une seule analyse)

**7.3: Ébauches en série -> `template_generated`.** Même méthode que 7.2, mais sur le « squelette » du texte (minuscules, chaque nombre remplacé par 0), pour que les fiches qui ne diffèrent que par un nom ou des chiffres se ressemblent. Les textes semblables à **0,7** ou plus sont reliés, sans regarder le sujet ; un texte est marqué s'il appartient à un groupe d'au moins **50 textes**.

**7.4: Texte répété -> `boilerplate`.** Une ligne d'au moins **4 mots** présente dans au moins **20 textes** de la langue est « répétée » ; un texte est marqué si au moins **50 %** de ses caractères viennent de lignes répétées.

**7.6: Contrôles sans drapeau** (comptés, avec exemples, pour le rapport qualité) : textes de moins de 5 et de moins de 20 mots, caractère de remplacement U+FFFD, texte mal décodé (`Ã©`, `â€™`…), messages d'erreur enregistrés à la place d'un article (« upstream connect error »…), textes qui commencent par une minuscule (nom du sujet perdu à la collecte).

Un texte peut recevoir plusieurs drapeaux. Cette analyse ne modifie pas la provenance : les drapeaux seront écrits en 7.7. Environ 25 minutes.

In [3]:
lancer("--analyse-qualite")
d = derniere("analyse_qualite")
aq = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{aq['textes']} textes : {aq['template_generated']} ébauches en série, {aq['boilerplate']} texte répété ; {aq['duree_s']} s")
print("7.6 :", aq["controles_7_6"])
tab = pd.DataFrame([{"langue": r["langue"], "textes": r["textes"],
                     "ébauches en série": r["t3"]["textes_marques"],
                     "% ébauches": round(100 * r["t3"]["textes_marques"] / r["textes"], 1) if r["textes"] else 0,
                     "groupes": r["t3"]["groupes"], "plus grand groupe": r["t3"]["plus_grand_groupe"],
                     "texte répété": r["t4"]["textes_marques"],
                     "% répété": round(100 * r["t4"]["textes_marques"] / r["textes"], 1) if r["textes"] else 0,
                     **{k: r["t6"]["compteurs"].get(k, 0) for k in ("moins_de_5_mots", "moins_de_20_mots",
                        "message_erreur", "texte_mal_decode", "debut_minuscule", "caractere_remplacement")}}
                    for r in aq["langues"]]).set_index("langue")
tab.sort_values("% ébauches", ascending=False)

2026-11-19T00:13:40 INFO 7.3 + 7.4 + 7.6 : 69 langue(s), 15 processus
2026-11-19T00:14:08 INFO zgh     12192 textes :    5996 ébauches en série (2 groupes),     55 texte répété ; < 20 mots 171, erreurs 0, mal décodé 0, début minuscule 35 (26 s)
2026-11-19T00:14:28 INFO ary     20321 textes :   10738 ébauches en série (11 groupes),      1 texte répété ; < 20 mots 610, erreurs 0, mal décodé 0, début minuscule 2 (47 s)
2026-11-19T00:14:29 INFO so      15572 textes :      82 ébauches en série (1 groupes),      4 texte répété ; < 20 mots 2296, erreurs 0, mal décodé 0, début minuscule 642 (48 s)
2026-11-19T00:14:31 INFO tn       7309 textes :       0 ébauches en série (0 groupes),      0 texte répété ; < 20 mots 200, erreurs 0, mal décodé 0, début minuscule 23 (50 s)
2026-11-19T00:14:33 INFO am      16286 textes :    2353 ébauches en série (5 groupes),     13 texte répété ; < 20 mots 6658, erreurs 0, mal décodé 0, début minuscule 5 (26 s)
2026-11-19T00:14:50 INFO yo      41803 textes :     3

,textes,ébauches en série,% ébauches,groupes,plus grand groupe,texte répété,% répété,moins_de_5_mots,moins_de_20_mots,message_erreur,texte_mal_decode,debut_minuscule,caractere_remplacement
langue,,,,,,,,,,,,,
tum,19880,12056,60.6,78,2021,51,0.3,47,11015,0,0,84,0
shi,10951,6089,55.6,4,5702,40,0.4,7,209,0,0,8966,0
ary,20321,10738,52.8,11,4829,1,0.0,7,610,0,0,2,0
zgh,12192,5996,49.2,2,5937,55,0.5,11,171,0,0,35,0
mg,104097,51088,49.1,101,11829,3795,3.6,56,6286,1,0,96,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ti,547,0,0.0,0,0,0,0.0,52,198,0,0,0,0
st,2710,0,0.0,0,0,0,0.0,16,521,0,0,19,0
tw,5641,0,0.0,0,0,0,0.0,13,664,0,0,31,0


In [4]:
# Exemples à relire pour une langue : groupes d'ébauches, lignes répétées, textes marqués « texte répété », contrôles 7.6
def exemples_qualite(code):
    r = next(x for x in aq["langues"] if x["langue"] == code)
    print(f"########## {code} : {r['t3']['textes_marques']} ébauches, {r['t4']['textes_marques']} texte répété")
    for e in r["t3"]["exemples"]:
        print(f"--- groupe de {e['taille']} ébauches")
        for x in e["extraits"]:
            print("   ", x[:160])
    print("--- lignes les plus répétées")
    for e in r["t4"]["lignes_les_plus_frequentes"][:8]:
        print(f"   {e['textes']:6} textes : {e['ligne'][:120]}")
    print("--- textes marqués « texte répété »")
    for e in r["t4"]["exemples"][:4]:
        print(f"   part {e['part']} : {e['extrait'][:160]}")
    for nom, ex in r["t6"]["exemples"].items():
        print(f"--- 7.6 {nom}")
        for e in ex[:3]:
            print("   ", e["extrait"][:160])

for code in ("wo", "tw", "ee", "fon", "yo", "ha"):
    exemples_qualite(code)

########## wo : 0 ébauches, 0 texte répété
--- lignes les plus répétées
      101 textes : Turu xam-xam wi
       43 textes : Jukki yi ci lonku
       29 textes : Tur wi ci yeneeni làkk
       16 textes : Dundu ak jaar-jaaram
       14 textes : Ci làkku ibrë la tur wi jóge.
       12 textes : Jukki yi ñu ko tudd
       10 textes : A lyga, Lituwaani: 0
        9 textes : Lees bind ci moom
--- textes marqués « texte répété »
--- 7.6 debut_minuscule
    sëriñ mooy tekki kilifa diinè lislaam. Am xamxam bu sell ak mëna def keemaan (ci doolè Yalla) mooy ñaari kalité yi ñuy g
    berat aydın ve fişkinin babaannesidir.bekir bir betadır.
    xam-xamu suuf si mooy li nuy wax (Jiyoloji), xam-xam la buy gëstu lépp lu aju ci suuf si kaw gi ak biir gi, ak doj yi (m
--- 7.6 message_erreur
     yi /  / misaalum xëtu HTML.html / HTM.htm /  / <!DOCTYPE html> /  / Turu Xët wi /  / Mbind mi!
    x / ASP.asp /  / <%@ Page Language="C#" %> /  / <!DOCTYPE html PUBLIC "- XHTML 1.0 Transitional / " /  / protec

In [5]:
for code in ("arz", "ar", "sw", "tum", "af", "fr"):
    exemples_qualite(code)

########## arz : 273872 ébauches, 277614 texte répété
--- groupe de 218225 ébauches
    NGC 4419 هيا مجره بتتبع كوكبة كوما برنيس و اللى اكتشفها ويليام هيرشيل و الاكتشاف كان بتاريخ 8 ابريل 1784. /  / معلومات المجره / الانزياح الأحمر: -0.0008. / السر
    COSMOS 1193005 هيا مجره بتتبع كوكبة سيكستانس. /  / معلومات المجره / الانزياح الأحمر: 0.0566. / السرعه الشعاعيه: 16489. / المطلع المستقيم: 150.512833. / الميل: 1
--- groupe de 20471 ébauches
    كوكس بحيره فى كندا. /  / جغرافيا / كوكس موجوده فى منطقه اداريه اسمها بريتيش كولومبيا. /  / شوف كمان / لستة اكبر بحيرات العالم حسب المساحه / لستة اكبر بحيرات الع
    پاتارا بحيره فى كندا. /  / جغرافيا / پاتارا موجوده فى منطقه اداريه اسمها اونتاريو, /  / شوف كمان / لستة اكبر بحيرات العالم حسب المساحه / لستة اكبر بحيرات العالم
--- groupe de 4678 ébauches
    2MASX J15595369+0359314 هيا مجره بتتبع كوكبة سيرپينس اللى هيا كوكبة فى نصف الكرة السماوية الشمالى . واحدة من 48 كوكبة حطها عالم الفلك بطليموس فى القرن التانى ، 
    SDSS J151225.08+105000.8 هيا م

## 7.5: Mauvaise langue -> `language_mismatch`

Chaque paragraphe d'au moins 5 mots est identifié par GlotLID v3 ; les paragraphes plus courts ne reçoivent pas de verdict.

- **Langue vérifiable** : un texte est marqué si moins de **50 %** des caractères de ses paragraphes identifiés sont dans la langue de l'édition. Chaque langue est jugée sur ses seules étiquettes (variétés de la même langue acceptées, aucune langue voisine).
- **Tigré** (modèle non fiable) et **langues absentes du modèle** (bole, pidgin ghanéen, igala, karekare, jju, nupe) : un texte est marqué seulement si au moins 50 % de ses caractères identifiés sont reconnus, avec une probabilité d'au moins **0,9**, comme anglais, français, portugais, espagnol, allemand ou arabe standard.

Le modèle occupe 1,7 Go de mémoire dans chaque processus : l'analyse tourne avec 4 processus (`qualite.processus_langue` dans `config.yaml` pour changer). Elle ne modifie pas la provenance.

In [3]:
lancer("--langue")
d = derniere("langue")
lg = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{lg['language_mismatch']} textes marqués language_mismatch sur {lg['textes']} "
      f"({100 * lg['language_mismatch'] / lg['textes']:.2f} %) ; {lg['duree_s']} s")
tab = pd.DataFrame([{"langue": r["langue"], "vérifiable": r["verifiable"], "textes": r["textes"],
                     "avec verdict": r["textes_avec_verdict"], "marqués": r["textes_marques"],
                     "% marqués": round(100 * r["textes_marques"] / r["textes_avec_verdict"], 1) if r["textes_avec_verdict"] else 0,
                     "langue trouvée (textes marqués)": ", ".join(f"{k} {v}" for k, v in list(r["etiquettes_des_textes_marques"].items())[:3])}
                    for r in lg["langues"]]).set_index("langue")
tab.sort_values("% marqués", ascending=False)

2026-11-21T20:01:32 INFO 7.5 Mauvaise langue (GlotLID model_v3.bin) : 69 langue(s), 1 processus (modèle chargé dans chacun)
2026-11-21T21:41:26 INFO ar    1332978 textes,  1325852 avec verdict,  346707 marqués  (5989 s) ; arz_Arab 227103, ary_Arab 88765, arb_Arab 10227
2026-11-21T22:36:21 INFO arz   1631258 textes,  1630618 avec verdict,   66755 marqués  (3295 s) ; arb_Arab 32137, eng_Latn 16671, arz_Arab 12977
2026-11-21T22:51:46 INFO ha     123651 textes,   123339 avec verdict,    1522 marqués  (926 s) ; eng_Latn 794, hau_Latn 426, dag_Latn 43
2026-11-21T23:09:03 INFO fr      99961 textes,    99861 avec verdict,    3247 marqués  (1036 s) ; fra_Latn 1453, eng_Latn 427, azj_Latn 194
2026-11-21T23:21:02 INFO af     131092 textes,   131013 avec verdict,   17240 marqués  (719 s) ; nld_Latn 6748, eng_Latn 5173, afr_Latn 3506
2026-11-21T23:27:58 INFO ig      56486 textes,    56434 avec verdict,     892 marqués  (416 s) ; eng_Latn 408, ibo_Latn 216, nno_Latn 71
2026-11-21T23:34:09 INFO sw   

,vérifiable,textes,avec verdict,marqués,% marqués,langue trouvée (textes marqués)
langue,,,,,,
ak,True,1,1,1,100.0,eng_Latn 1
kus,True,1721,1716,1617,94.2,"dag_Latn 1149, gur_Latn 148, eng_Latn 78"
ng,True,20,13,12,92.3,"efi_Latn 3, nbl_Latn 2, xho_Latn 2"
kg,True,2256,2241,1450,64.7,"ktu_Latn 1386, lin_Latn 24, kng_Latn 7"
sg,True,505,337,92,27.3,"kmr_Latn 19, fra_Latn 13, sag_Latn 7"
...,...,...,...,...,...,...
nup,False,1609,1607,1,0.1,eng_Latn 1
aa,False,0,0,0,0.0,
kr,True,0,0,0,0.0,


In [4]:
# Exemples de textes marqués « mauvaise langue »
def exemples_langue(*codes):
    for r in lg["langues"]:
        if r["langue"] in codes:
            print(f"########## {r['langue']} : {r['textes_marques']} marqués sur {r['textes_avec_verdict']}")
            for e in r["exemples"][:6]:
                print(f"   [{e['etiquette_principale']}, part attendue {e['part_attendue']}] {e['extrait'][:160]}")

exemples_langue("wo", "tw", "ee", "fon", "yo", "ha", "rn", "kus", "kg", "ar", "arz", "kai", "tig")

########## ar : 346707 marqués sur 1325852
   [ary_Arab, part attendue 0.32] الرِّيَاضِيَّاتُ هي مجموعة من المعارف المجردة الناتجة عن الاستنتاجات المنطقية المطبقة على مُختلف الكائنات الرياضية مثل المجموعات، والأعداد، والأشكال والبنيات وا
   [deu_Latn, part attendue 0.34] بسام طيبي هو خبير سوري في مجال العلوم السياسية مقيم في ألمانيا منذ عام 1962 وأصبح مواطنا ألمانيا في عام 1976. وُلِد بسام طيبي في 4 أبريل 1944 في العاصمة السورية
   [ary_Arab, part attendue 0.0] النشيد الشريف هو النشيد الوطني الرسمي للمغرب منذ الاستقلال عن الحماية سنة 1956. كتبه علي الصقلي الحسيني ولحنه ليو مورغان. /  / قصة النشيد الوطني المغربي / النشي
   [ary_Arab, part attendue 0.37] اللغة الأمازيغية (الاسم الأمازيغي: ، نيو-تيفيناغ: ⵜⴰⵎⴰⵣⵉⵖⵜ، تيفيناغ طارقية: ⵜⴰⵎⴰⵣⵉⵗⵜ، ⵝⴰⵎⴰⵣⵉⵗⵝ، بالحرف اللاتيني: Tamaziɣt) هي لغة رسمية في كل من المغرب والجزائر
   [ars_Arab, part attendue 0.0] الظهر تأخذ أكثر من معنى حسب اللفظ: /  / الظهر، زوال الشمس أو منتصف النهار / ظهر، الجزء الخلفي والمتوسط من الجسد، يوجد فيه العمود الفقري والحبل ال

## 7.7: Écriture finale des `quality_flags`

Combine les résultats retenus : 7.2 (`near_duplicate`), 7.3+7.4 (`template_generated`, `boilerplate`), 7.5 (`language_mismatch`). Écrit une **nouvelle** provenance (le corpus normalisé et sa provenance d'origine ne sont jamais modifiés). Les 2 vrais messages d'erreur de collecte (motif resserré) ne reçoivent pas de drapeau : ils sont listés à part dans `exclusions_export.jsonl`, prêts pour la tâche 8 (raison « erreur de collecte »). Des contrôles automatiques vérifient qu'aucune langue n'a de décalage corpus/provenance ni d'indice de drapeau hors limites.

In [8]:
lancer("--ecrire-drapeaux")
d = derniere("ecriture_drapeaux")
eb = json.loads((d / "bilan.json").read_text(encoding="utf-8"))
print(f"{eb['textes']} textes ; au moins un drapeau : {eb['au_moins_un_drapeau']} ({100 * eb['au_moins_un_drapeau'] / eb['textes']:.2f} %) ; "
      f"plusieurs drapeaux : {eb['plusieurs_drapeaux']} ; erreurs de collecte (export) : {eb['erreurs_collecte_total']} ; {eb['duree_s']} s")
print("anomalies :", eb["anomalies"] or "aucune")
tab = pd.DataFrame([{"langue": r["langue"], "textes": r["textes"], "near_duplicate": r["near_duplicate"],
                     "template_generated": r["template_generated"], "boilerplate": r["boilerplate"],
                     "language_mismatch": r["language_mismatch"], ">=1 drapeau": r["au_moins_un_drapeau"],
                     "% >=1 drapeau": round(100 * r["au_moins_un_drapeau"] / r["textes"], 2) if r["textes"] else 0}
                    for r in eb["langues"]]).set_index("langue")
tab.sort_values("% >=1 drapeau", ascending=False)

2026-11-22T07:04:53 INFO 7.7 Écriture des quality_flags : 69 langue(s), 15 processus ; sources near_duplicate=20261118T073610Z_quasi_doublons, template_generated/boilerplate=20261119T001340Z_analyse_qualite, language_mismatch=20261121T200132Z_langue
2026-11-22T07:04:57 INFO zgh     12192 textes :     31 near_duplicate,   5996 template_generated,     55 boilerplate,    308 language_mismatch ;   6361 avec >=1 drapeau,  0 erreur(s) collecte  (3 s)
2026-11-22T07:05:00 INFO ary     20321 textes :    254 near_duplicate,  10738 template_generated,      1 boilerplate,    191 language_mismatch ;  10952 avec >=1 drapeau,  0 erreur(s) collecte  (6 s)
2026-11-22T07:05:00 INFO tn       7309 textes :     17 near_duplicate,      0 template_generated,      0 boilerplate,    163 language_mismatch ;    180 avec >=1 drapeau,  0 erreur(s) collecte  (6 s)
2026-11-22T07:05:00 INFO am      16286 textes :     14 near_duplicate,   2353 template_generated,     13 boilerplate,   2037 language_mismatch ;   2535 a

,textes,near_duplicate,template_generated,boilerplate,language_mismatch,>=1 drapeau,% >=1 drapeau
langue,,,,,,,
ak,1,0,0,0,1,1,100.00
kus,1721,9,0,0,1617,1620,94.13
tum,19880,6,12056,51,2394,13793,69.38
kg,2256,46,58,0,1450,1510,66.93
ng,20,0,0,0,12,12,60.00
...,...,...,...,...,...,...,...
nqo,1726,8,0,0,4,11,0.64
aa,0,0,0,0,0,0,0.00
kr,0,0,0,0,0,0,0.00


In [9]:
# Les textes à écarter à l'export (tâche 8), raison "erreur de collecte" — aucun drapeau qualité ici
excl = pd.read_json(d / "exclusions_export.jsonl", lines=True) if (d / "exclusions_export.jsonl").stat().st_size else None
excl

,langue,source_url,text_sha256_collecte,raison,extrait
0,af,https://af.wikipedia.org/w/index.php?oldid=286...,756f6b4e49037945604c58f58db5959caadbae88247dc7...,erreur de collecte,regtig om die dokument te sien nie. / 500 Inte...
1,ar,https://ar.wikipedia.org/w/index.php?oldid=753...,e4cf3e8d047cccdb09e407ba560d7479c78834710f43d0...,erreur de collecte,ص بالانجليزية / المعنى / 500 / خطأ خادم داخلي ...
2,ha,https://ha.wikipedia.org/w/index.php?oldid=499242,0fdcd4f2c341fd362aa7b4b1331cbe0cb8c34790625cb7...,erreur de collecte,upstream connect error or disconnect/reset bef...
3,ig,https://ig.wikipedia.org/w/index.php?oldid=142498,0fdcd4f2c341fd362aa7b4b1331cbe0cb8c34790625cb7...,erreur de collecte,upstream connect error or disconnect/reset bef...
